# test-B — C2: Docling layout-aware, table-atomic chunking

answering: **gpt-4o-mini**, embedding: **text-embedding-3-small**.

Means over 3 repeats of 150 in-scope and 20 out-of-scope questions. Reads `answers-r*.json` and `judge-r*.json` in this folder; nothing here calls an LLM.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "code"))  # test/code
import imka_eval as E

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUN = "test-B"
rows = E.run_rows(RUN)
in_scope = rows[rows.group != "out_of_scope"]

## Documents

The five documents the run searched (pinned in `documents.json`).

In [2]:
docs = E._read_json(E.run_dir(RUN) / "documents.json")["documents"]
pd.DataFrame([{"document": d["document_name"][:45], "document_id": d["document_id"], "chunks": d.get("chunk_count"), "captioning": (d.get("models") or {}).get("captioning", "-"), "embedding": (d.get("embedding") or {}).get("model", E.DEFAULT_EMBEDDING)} for d in docs])

,document,document_id,chunks,captioning,embedding
0,ABB Manual for Induction Motors and Generator,fdb5c8bb-ba46-40f5-b9a2-e30841e3a2b4,478,-,text-embedding-3-small
1,EMSA Diesel Generator manual-en.pdf,5e59d8ad-dde6-4c8b-9ca9-471960f2a127,285,-,text-embedding-3-small
2,Kohler Generator Manual.pdf,ad188559-78e5-49df-a136-fd909eb8407b,294,-,text-embedding-3-small
3,Nidec Motor Manual.pdf,ecc5861a-047f-4623-bbf0-679bffbfde8f,116,-,text-embedding-3-small
4,Siemens_Manual_Vertical_Induction_Motors_EN.p,23b6d1a9-0fcc-42f1-85f8-ec43c7060425,119,-,text-embedding-3-small


## Results by question group

In [3]:
E.summary(runs=(RUN,), by="group")

n  hit@1  hit@3  hit@5  reciprocal_rank  correct_pdf@5  correct_pdf@1  candidates_hit  faithfulness  answer_relevancy  context_recall  answer_accuracy  refused
run    group                                                                                                                                                                           
test-B tables          30  0.778  0.944  1.000            0.871          1.000          0.967           1.000         0.693             0.875           0.956            0.833      NaN
       cross_document  30  0.878  0.944  0.967            0.914          1.000          0.900           1.000         0.729             0.892           0.952            0.892      NaN
       topical         30  0.711  0.867  0.922            0.799          0.967          0.956           0.967         0.795             0.767           0.815            0.794      NaN
       paraphrase      35  0.610  0.829  0.905            0.736          1.000          0.933           0.981         0.680             0.803           0.852            0.757      NaN
       figures         25  0.733  0.867  0.880            0.792          1.000          0.973           0.947         0.849             0.412           0.397            0.380      NaN
       out_of_scope    20    NaN    NaN    NaN              NaN            NaN            NaN             NaN           NaN               NaN             NaN              NaN      1.0

In [4]:
E.summary(runs=(RUN,), by=None)

,n,hit@1,hit@3,hit@5,reciprocal_rank,correct_pdf@5,correct_pdf@1,candidates_hit,faithfulness,answer_relevancy,context_recall,answer_accuracy,refused
run,,,,,,,,,,,,,
test-B,170,0.738,0.889,0.936,0.82,0.993,0.944,0.98,0.744,0.763,0.81,0.744,1.0


## Stability across the three repeats

In [5]:
in_scope.groupby("repeat")[["hit@5", "context_recall", "answer_accuracy"]].mean().round(3)

,hit@5,context_recall,answer_accuracy
repeat,,,
1,0.933,0.810,0.748
2,0.940,0.814,0.740
3,0.933,0.804,0.743


## Out-of-scope questions

Share of the 20 out-of-scope questions declined, per repeat.

In [6]:
rows[rows.group == "out_of_scope"].groupby("repeat")["refused"].mean().round(3)

repeat
1    1.0
2    1.0
3    1.0
Name: refused, dtype: float64

## Lowest answer accuracy

The ten in-scope questions with the lowest accuracy, averaged over the repeats, with the first repeat's answer.

In [7]:
answers = {r["question"]: r.get("answer") or "" for r in E.load_answers(RUN, 1)["results"]}
worst = (in_scope.groupby(["question", "group", "document_name"])["answer_accuracy"].mean()
         .sort_values().head(10).reset_index())
worst["document_name"] = worst["document_name"].str[:20]
worst["answer (repeat 1)"] = worst["question"].map(lambda q: answers.get(q, "")[:160])
worst

,question,group,document_name,answer_accuracy,answer (repeat 1)
0,"As a thumb rule, how often does ABB say an ins...",topical,ABB Manual for Induc,0.0,The context does not specify a frequency or sp...
1,How does ABB calculate equivalent operating ho...,topical,ABB Manual for Induc,0.0,The context does not provide specific details ...
2,How does ABB calculate equivalent operating ho...,topical,ABB Manual for Induc,0.0,The provided context does not include specific...
3,How do I work out equivalent running hours for...,paraphrase,ABB Manual for Induc,0.0,To work out equivalent running hours (Eq. h) f...
4,"From Figure 42 of the ABB manual, at 18 brushe...",figures,ABB Manual for Induc,0.0,The specific maximum cleaning interval for 18 ...
5,"From Figure 42 of the ABB manual, at 6 brushes...",figures,ABB Manual for Induc,0.0,The context does not provide specific numerica...
6,"In Fig. 27 of the Kohler KD27V12 manual, appro...",figures,Kohler Generator Man,0.0,The context does not provide information about...
7,"In the ABB induction machine manual, what is t...",tables,ABB Manual for Induc,0.0,The Pt-100 resistance at exactly 100°C is 212....
8,"In Fig. 27 of the Kohler KD27V12 manual, what ...",figures,Kohler Generator Man,0.0,The context does not provide specific values f...
9,"From Figure 38 of the ABB manual, approximatel...",figures,ABB Manual for Induc,0.0,The context does not provide the exact value o...


## Answering cost and time per question

OpenAI models are priced per token; Qwen per second of model working time (llama-server's prompt + generation time) at `E.GPU_PRICE_PER_HOUR`. RunPod queueing, cold starts and job overhead are shown (`runpod_exec_s`, `runpod_delay_s`) but not charged. `wall_s` is the question's end-to-end time as the evaluation saw it.

In [8]:
E.answering_cost((RUN,)).T

run,test-B
answers,510
calls/q,3.0
prompt_tokens/q,2392
completion_tokens/q,173
model_s/q,None
runpod_exec_s/q,None
runpod_delay_s/q,None
wall_s/q (median),4.6
"cost per 1,000 questions ($)",0.46


## Judge spend (Anthropic)

In [9]:
spend = E.judge_spend((RUN,))
spend.groupby("model")[["calls", "input_tokens", "output_tokens", "cost_usd"]].sum()

,calls,input_tokens,output_tokens,cost_usd
model,,,,
claude-haiku-5-5,2760,5742750,764743,0.9566
claude-sonnet-5-5,900,1169261,9000,2.4285
